# 🎫 FUTURE_ML_02: Support Ticket Classification
> **Internship ML Project** | NLP + Scikit-learn | Beginner-Friendly

---
### 📋 What We're Building
A system that reads a customer support ticket and automatically:
1. **Classifies** it into a category (Billing, Technical Issue, etc.)
2. **Assigns a priority** level (High / Medium / Low)

### 🗺️ Notebook Roadmap
| Step | Task |
|------|------|
| 1 | Setup & Imports |
| 2 | Load & Explore Data |
| 3 | Text Preprocessing |
| 4 | Feature Extraction (TF-IDF) |
| 5 | Train Models |
| 6 | Evaluate Models |
| 7 | Visualize Results |
| 8 | Predict New Tickets |
| 9 | Future Improvements |

## Step 1 — Setup & Imports

In [ ]:
# ── Standard Library ─────────────────────────────────────────
import os, re, sys, pickle, warnings
warnings.filterwarnings('ignore')        # keep output clean

# ── Data Handling ─────────────────────────────────────────────
import numpy  as np
import pandas as pd

# ── Visualizations ────────────────────────────────────────────
import matplotlib.pyplot    as plt
import matplotlib.gridspec  as gridspec
import seaborn              as sns
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

# ── Scikit-learn: Feature Extraction ──────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# ── Scikit-learn: Models ───────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes  import MultinomialNB
from sklearn.pipeline     import Pipeline

# ── Scikit-learn: Utilities ────────────────────────────────────
from sklearn.model_selection   import train_test_split, cross_val_score
from sklearn.preprocessing     import LabelEncoder
from sklearn.metrics           import (accuracy_score, classification_report,
                                       confusion_matrix, ConfusionMatrixDisplay)

# Add project root to path so we can import from src/
sys.path.insert(0, os.path.abspath('..'))

print('✅ All imports successful!')
print(f'   NumPy   {np.__version__}')
print(f'   Pandas  {pd.__version__}')
import sklearn; print(f'   Sklearn {sklearn.__version__}')

## Step 2 — Load & Explore the Dataset

In [ ]:
# ── Load CSV ──────────────────────────────────────────────────
DATA_PATH = '../data/support_tickets.csv'

if not os.path.exists(DATA_PATH):
    print('Generating dataset...')
    os.system(f'python ../data/generate_dataset.py')

df = pd.read_csv(DATA_PATH)

print(f'Dataset shape : {df.shape}  (rows × columns)')
print(f'Columns       : {list(df.columns)}')
df.head()

In [ ]:
# ── Check for missing values ──────────────────────────────────
print('Missing values per column:')
print(df.isnull().sum())

print(f'\nData types:')
print(df.dtypes)

In [ ]:
# ── Distribution of classes ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Class Distribution in Dataset', fontsize=14, fontweight='bold')

# Category
cat_counts = df['category'].value_counts()
axes[0].barh(cat_counts.index, cat_counts.values,
             color=sns.color_palette('Set2', len(cat_counts)))
axes[0].set_title('Ticket Categories')
axes[0].set_xlabel('Count')
for i, v in enumerate(cat_counts.values):
    axes[0].text(v + 0.5, i, str(v), va='center', fontweight='bold')

# Priority
pri_counts = df['priority'].value_counts()
colors = {'High': '#DC2626', 'Medium': '#D97706', 'Low': '#16A34A'}
axes[1].bar(pri_counts.index,
            pri_counts.values,
            color=[colors[p] for p in pri_counts.index])
axes[1].set_title('Priority Levels')
axes[1].set_ylabel('Count')
for i, (p, v) in enumerate(zip(pri_counts.index, pri_counts.values)):
    axes[1].text(i, v + 1, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── Ticket length analysis ────────────────────────────────────
df['ticket_length'] = df['ticket_text'].apply(len)
df['word_count']    = df['ticket_text'].apply(lambda x: len(x.split()))

print(df[['ticket_length', 'word_count']].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['ticket_length'].hist(bins=30, ax=axes[0], color='#2563EB', alpha=0.8)
axes[0].set_title('Ticket Character Length Distribution')
axes[0].set_xlabel('Number of Characters')

df['word_count'].hist(bins=20, ax=axes[1], color='#7C3AED', alpha=0.8)
axes[1].set_title('Ticket Word Count Distribution')
axes[1].set_xlabel('Number of Words')

plt.tight_layout()
plt.show()

## Step 3 — Text Preprocessing

Raw text has noise that can hurt our model. We need to:
- Make everything **lowercase** (so `Crash` and `crash` are the same)
- Remove **punctuation** and **numbers**
- Remove **stopwords** (the, is, a, an…) — they add no meaning
- Keep only **meaningful words**

This is called **text normalization**.

In [ ]:
# ── Stopwords list (built-in, no internet needed) ─────────────
STOPWORDS = {
    'i','me','my','myself','we','our','ours','ourselves','you','your','yours',
    'yourself','yourselves','he','him','his','himself','she','her','hers',
    'herself','it','its','itself','they','them','their','theirs','themselves',
    'what','which','who','whom','this','that','these','those','am','is','are',
    'was','were','be','been','being','have','has','had','having','do','does',
    'did','doing','a','an','the','and','but','if','or','because','as','until',
    'while','of','at','by','for','with','about','against','between','into',
    'through','during','before','after','above','below','to','from','up','down',
    'in','out','on','off','over','under','again','further','then','once','here',
    'there','when','where','why','how','all','both','each','few','more','most',
    'other','some','such','no','nor','not','only','own','same','so','than','too',
    'very','s','t','can','will','just','should','now','d','ll','m','o','re','ve',
    'y','hi','hello','dear','please','thank','thanks','good','morning','need',
    'want','help','like','know','make','use','using','used','still','already',
    'back','time','day','days','week','try','tried','trying','also','every',
    'even','get','got','may','might','much','one','two','three','us','would',
    'could',
}

def clean_text(text):
    """Clean one ticket string step by step."""
    if not isinstance(text, str):
        return ''
    text = text.lower()                          # 1. lowercase
    text = re.sub(r'http\S+|www\.\S+', ' ', text) # 2. remove URLs
    text = re.sub(r'\S+@\S+', ' ', text)          # 3. remove emails
    text = re.sub(r'[^a-z\s]', ' ', text)         # 4. keep only letters
    text = re.sub(r'\s+', ' ', text).strip()      # 5. collapse spaces
    tokens = [w for w in text.split()
              if w not in STOPWORDS and len(w) > 2]  # 6. remove stopwords
    return ' '.join(tokens)

# Apply to the whole dataframe
df['clean_text'] = df['ticket_text'].apply(clean_text)

# Show 5 before/after examples
print('Before/After Preprocessing Examples:\n')
for i in range(5):
    print(f'BEFORE: {df["ticket_text"].iloc[i][:90]}')
    print(f'AFTER : {df["clean_text"].iloc[i][:90]}')
    print('─' * 70)

## Step 4 — Feature Extraction: TF-IDF

ML models cannot read words — they need **numbers**.

**TF-IDF** (Term Frequency × Inverse Document Frequency) converts text into a matrix of numbers where:
- Each **row** = one ticket
- Each **column** = one unique word
- Each **value** = how important that word is in that ticket

Common words (like "the") get LOW scores. Rare, specific words (like "refund", "crash") get HIGH scores.

In [ ]:
# ── Demonstrate TF-IDF ────────────────────────────────────────
demo_docs = [
    'account locked cannot login password reset',
    'charged twice billing invoice refund',
    'server crash app broken error',
]

demo_tfidf = TfidfVectorizer()
demo_matrix = demo_tfidf.fit_transform(demo_docs)

demo_df = pd.DataFrame(
    demo_matrix.toarray().round(3),
    columns=demo_tfidf.get_feature_names_out(),
    index=['Account ticket', 'Billing ticket', 'Technical ticket']
)
print('TF-IDF Matrix (demo with 3 tickets):')
demo_df

In [ ]:
# ── Label Encoding ────────────────────────────────────────────
# Convert string labels to integers (required by sklearn)

cat_encoder = LabelEncoder()
pri_encoder = LabelEncoder()

df['category_encoded'] = cat_encoder.fit_transform(df['category'])
df['priority_encoded'] = pri_encoder.fit_transform(df['priority'])

print('Category encoding:')
for i, cls in enumerate(cat_encoder.classes_):
    print(f'  {i} → {cls}')

print('\nPriority encoding:')
for i, cls in enumerate(pri_encoder.classes_):
    print(f'  {i} → {cls}')

## Step 5 — Train/Test Split & Model Training

In [ ]:
# ── Train/Test Split ──────────────────────────────────────────
# 80% training data, 20% test data
# stratify= makes sure all classes appear proportionally in both splits

X = df['clean_text']
y_cat = df['category']
y_pri = df['priority']

X_train, X_test, yc_train, yc_test, yp_train, yp_test = train_test_split(
    X, y_cat, y_pri,
    test_size=0.20,
    random_state=42,
    stratify=y_cat
)

print(f'Train samples : {len(X_train)}')
print(f'Test  samples : {len(X_test)}')
print(f'\nTrain category distribution:')
print(yc_train.value_counts())

In [ ]:
# ── Build Pipeline ────────────────────────────────────────────
# A Pipeline chains steps so we don't repeat code:
#   raw text → TF-IDF → Classifier

def make_pipeline(classifier='logistic'):
    clf = (LogisticRegression(max_iter=1000, random_state=42)
           if classifier == 'logistic' else MultinomialNB())
    return Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),   # unigrams + bigrams
            sublinear_tf=True,    # log scaling
        )),
        ('clf', clf)
    ])

# ── Train BOTH models ─────────────────────────────────────────
cat_pipe = make_pipeline('logistic')
cat_pipe.fit(X_train, yc_train)

pri_pipe = make_pipeline('logistic')
pri_pipe.fit(X_train, yp_train)

print('✅ Both models trained!')

## Step 6 — Evaluation Metrics

**Accuracy** = correct predictions / total predictions

**Precision** = of everything labeled class X, how many actually were X?

**Recall** = of all actual class X items, how many did we catch?

**F1 Score** = harmonic mean of Precision and Recall (best single metric)

In [ ]:
# ── Predict on test set ───────────────────────────────────────
yc_pred = cat_pipe.predict(X_test)
yp_pred = pri_pipe.predict(X_test)

cat_acc = accuracy_score(yc_test, yc_pred)
pri_acc = accuracy_score(yp_test, yp_pred)

print(f'Category Accuracy : {cat_acc:.4f}  ({cat_acc*100:.1f}%)')
print(f'Priority Accuracy : {pri_acc:.4f}  ({pri_acc*100:.1f}%)')

# ── Cross Validation (more reliable estimate) ─────────────────
cat_cv = cross_val_score(cat_pipe, X, y_cat, cv=5)
pri_cv = cross_val_score(pri_pipe, X, y_pri, cv=5)

print(f'\nCategory CV (5-fold): {cat_cv.mean():.4f} ± {cat_cv.std():.4f}')
print(f'Priority  CV (5-fold): {pri_cv.mean():.4f} ± {pri_cv.std():.4f}')

In [ ]:
# ── Full Classification Report ────────────────────────────────
print('═'*60)
print('CATEGORY CLASSIFICATION REPORT')
print('═'*60)
print(classification_report(yc_test, yc_pred))

print('═'*60)
print('PRIORITY PREDICTION REPORT')
print('═'*60)
print(classification_report(yp_test, yp_pred))

## Step 7 — Visualizations

In [ ]:
# ── Confusion Matrix: Category ────────────────────────────────
cat_labels = sorted(df['category'].unique())
cm_cat = confusion_matrix(yc_test, yc_pred, labels=cat_labels)

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(cm_cat, display_labels=cat_labels)
disp.plot(ax=ax, cmap='Blues', colorbar=True)
ax.set_title('Confusion Matrix — Category Classification',
             fontsize=13, fontweight='bold', pad=12)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion Matrix: Priority ────────────────────────────────
pri_labels = ['High', 'Low', 'Medium']
cm_pri = confusion_matrix(yp_test, yp_pred, labels=pri_labels)

fig, ax = plt.subplots(figsize=(6, 5))
disp2 = ConfusionMatrixDisplay(cm_pri, display_labels=pri_labels)
disp2.plot(ax=ax, cmap='Reds', colorbar=True)
ax.set_title('Confusion Matrix — Priority Prediction',
             fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Top TF-IDF Features per Category ─────────────────────────
# Which words does the model consider most important for each class?

feature_names = cat_pipe.named_steps['tfidf'].get_feature_names_out()
clf           = cat_pipe.named_steps['clf']

fig, axes = plt.subplots(1, len(cat_labels), figsize=(18, 5))
fig.suptitle('Top 10 Most Informative Words per Category',
             fontsize=13, fontweight='bold')

palette = ['#2563EB','#16A34A','#D97706','#DC2626','#7C3AED']

for i, (label, ax) in enumerate(zip(cat_labels, axes)):
    top_idx  = clf.coef_[i].argsort()[-10:]
    top_words = [feature_names[j] for j in top_idx]
    top_vals  = clf.coef_[i][top_idx]

    ax.barh(top_words, top_vals, color=palette[i], alpha=0.85)
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.set_xlabel('Coef Weight')
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ── Model Accuracy Comparison ─────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))

tasks = ['Category\nClassification', 'Priority\nPrediction']
accs  = [cat_acc * 100, pri_acc * 100]
clrs  = ['#2563EB', '#DC2626']

bars = ax.bar(tasks, accs, color=clrs, width=0.4,
              edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            f'{val:.1f}%', ha='center', fontsize=13, fontweight='bold',
            color=bar.get_facecolor())

ax.set_ylim(0, 115)
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Model Accuracy Comparison', fontweight='bold', fontsize=13)
ax.spines[['top','right']].set_visible(False)
ax.yaxis.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Step 8 — Predict New Tickets

In [ ]:
def predict_ticket(text, cat_model, pri_model):
    """Predict category and priority for a raw ticket string."""
    cleaned  = clean_text(text)
    category = cat_model.predict([cleaned])[0]
    priority = pri_model.predict([cleaned])[0]

    cat_prob = dict(zip(cat_model.classes_,
                        cat_model.predict_proba([cleaned])[0]))
    pri_prob = dict(zip(pri_model.classes_,
                        pri_model.predict_proba([cleaned])[0]))

    icons = {'High': '🔴', 'Medium': '🟡', 'Low': '🟢'}
    print(f'\n📩 Ticket   : {text[:90]}')
    print(f'🏷️  Category  : {category}')
    print(f'{icons[priority]} Priority  : {priority}')
    print(f'   Confidence → Category: {max(cat_prob.values()):.1%}  '
          f'Priority: {max(pri_prob.values()):.1%}')

# ── Test with new tickets ─────────────────────────────────────
new_tickets = [
    'I cannot login to my account. Password reset emails do not arrive.',
    'I was billed twice for my subscription. Please refund the extra charge.',
    'The app crashes immediately when I try to open the dashboard.',
    'What is the difference between the Basic and Pro plans?',
    'My account was suspended without any warning. I need urgent help!',
]

print('=' * 65)
print('SAMPLE PREDICTIONS ON UNSEEN TICKETS')
print('=' * 65)
for ticket in new_tickets:
    predict_ticket(ticket, cat_pipe, pri_pipe)

In [ ]:
# ── Try your own ticket! ──────────────────────────────────────
my_ticket = 'I need a full refund for my annual subscription I purchased by mistake.'

predict_ticket(my_ticket, cat_pipe, pri_pipe)

In [ ]:
# ── Save models to disk ───────────────────────────────────────
os.makedirs('../models', exist_ok=True)

with open('../models/category_pipeline.pkl', 'wb') as f:
    pickle.dump(cat_pipe, f)

with open('../models/priority_pipeline.pkl', 'wb') as f:
    pickle.dump(pri_pipe, f)

print('✅ Models saved!')
print('   models/category_pipeline.pkl')
print('   models/priority_pipeline.pkl')

## Step 9 — Future Improvements

The model currently achieves excellent accuracy on this dataset. Here are ideas to push it even further in real-world settings:

### 🔧 Short-term (1–2 weeks)
| Idea | Benefit |
|------|--------|
| Add more real-world data | More variety = more robust model |
| Try Random Forest / SVM | May outperform Logistic Regression |
| Hyperparameter tuning (GridSearchCV) | Find optimal model settings |
| Add stemming/lemmatization | Reduce vocabulary size |

### 🚀 Medium-term (1–2 months)
| Idea | Benefit |
|------|--------|
| Use BERT / DistilBERT embeddings | Far better text understanding |
| Deploy as REST API (FastAPI/Flask) | Makes it production-ready |
| Add multi-label classification | One ticket → multiple tags |
| Confidence threshold alerts | Flag low-confidence predictions for human review |

### 🌟 Long-term (3+ months)
| Idea | Benefit |
|------|--------|
| Active learning loop | Model improves from corrections |
| Multi-language support | Handle tickets in Spanish, French, etc. |
| Sentiment analysis | Detect angry/frustrated customers |
| Auto-response generation | LLM generates suggested replies |